# USD SOFR Swap Curve Bootstrapping — Live FRED Data

**Goal:** bootstrap a USD SOFR discount / zero curve from market rates, with all
FRED-published inputs fetched live via the FRED REST API.

**Method & derivations:** see the companion document
[`SOFR_Bootstrapping_Methodology.md`](./SOFR_Bootstrapping_Methodology.md)
(research notes, hand-calculated worked example, implementation plan).

## Data sources — read this first

| Curve segment | Instrument | Source | Caveat |
|---|---|---|---|
| Overnight | SOFR fixing | FRED `SOFR` | exact |
| 1M / 3M / 6M | 30/90/180-day compounded SOFR averages | FRED `SOFR30DAYAVG` / `SOFR90DAYAVG` / `SOFR180DAYAVG` | **backward-looking** averages used as forward-looking term-rate approximations |
| 1Y – 30Y | SOFR OIS par swap rates | **not on FRED** → either `MANUAL` quotes (Bloomberg/CME) or `TREASURY_PROXY`: FRED `DGS*` Treasury yields + configurable swap-spread adjustment | proxy quotes are approximations; spreads are user inputs |

Because SOFR OIS swaps both *pay* compounded SOFR and are *discounted* at SOFR, a single
curve serves as projection and discount curve (single-curve framework).

## Methodology in four formulas

**1. Par condition** for a spot-starting OIS with fixed rate $S_N$, annual fixed coupon dates
$T_1<\dots<T_N$ and Act/360 accruals $\tau_i$ (floating leg telescopes to $1-P(0,T_N)$):

$$S_N \sum_{i=1}^{N} \tau_i\,P(0,T_i) \;=\; 1 - P(0,T_N)$$

**2. Bootstrap recursion** (all earlier $P$ known ⇒ closed form for the newest node):

$$P(0,T_N) = \frac{1 - S_N \sum_{i=1}^{N-1} \tau_i\,P(0,T_i)}{1 + S_N\,\tau_N}$$

**3. Gap tenors** (e.g. the 4Y coupon inside the 5Y swap when 4Y is not quoted): treat the new
node DF as unknown $x$, interpolate interior coupon DFs **log-linearly in $\ln P$**
(⇔ piecewise-constant overnight forwards), and solve
$g(x)=S_N\sum\tau_i P_x(0,T_i)-(1-x)=0$ with Brent's method — $g$ is strictly increasing in
$x$, so the root is unique.

**4. Outputs:** zero rate $z(T)=-\ln P(0,T)/T$ (continuous, Act/365) and forwards
$f(T_a,T_b)=\bigl(P(0,T_a)/P(0,T_b)-1\bigr)/\tau(T_a,T_b)$.

**Correctness certificate:** re-price every input swap off the finished curve; each must
recover its input quote to machine precision (asserted below).

In [ ]:
# =============================================================================
# SECTION 1 — Imports & user configuration (the ONLY cell you need to edit)
# =============================================================================
import math
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
import requests                      # HTTP client for the FRED REST API
import matplotlib.pyplot as plt
from scipy.optimize import brentq    # 1-D root finder for gap-tenor bootstrapping

# --- FRED API access ---------------------------------------------------------
# Free key from https://fred.stlouisfed.org/docs/api/api_key.html
# (Best practice outside a demo: keep the key in an environment variable,
#  e.g. os.environ["FRED_API_KEY"], instead of hard-coding it.)
FRED_API_KEY = "d7230d655de6669a15d47c2c25bfe847"
FRED_BASE    = "https://api.stlouisfed.org/fred/series/observations"

# --- Reference date ----------------------------------------------------------
# "latest"  -> use the most recent published observation of every series
# "YYYY-MM-DD" -> build the curve as of that (or the nearest earlier) business day
REFERENCE_DATE = "latest"

# --- Long-end swap quotes: choose ONE of two sources -------------------------
# "MANUAL"         : you paste true SOFR OIS par swap rates (Bloomberg / CME) below.
#                    This is the correct production-quality path.
# "TREASURY_PROXY" : fully live fallback — fetch Treasury constant-maturity yields
#                    (FRED DGS1..DGS30) and add a per-tenor swap-spread adjustment,
#                    because FRED does NOT publish SOFR OIS par swap rates.
SWAP_QUOTE_SOURCE = "TREASURY_PROXY"

# Used only when SWAP_QUOTE_SOURCE == "MANUAL".  Percent, annual/Act360 par rates.
MANUAL_SWAP_QUOTES_PCT = {
    1: 3.98, 2: 4.01, 3: 4.04, 5: 4.08, 7: 4.14, 10: 4.23, 20: 4.55, 30: 4.42,
}

# Used only when SWAP_QUOTE_SOURCE == "TREASURY_PROXY".
# SOFR swap spread (swap MINUS Treasury) in basis points, by tenor.
# ILLUSTRATIVE values only — recent-era spreads are negative and widen with
# maturity.  Update from a market snapshot before relying on the long end.
PROXY_SWAP_SPREAD_BP = {
    1: -5, 2: -20, 3: -25, 5: -30, 7: -38, 10: -45, 20: -65, 30: -80,
}

SWAP_TENORS_Y = [1, 2, 3, 5, 7, 10, 20, 30]   # bootstrapped long-end pillars

pd.set_option("display.float_format", lambda v: f"{v:,.6f}")
print(f"Config OK — quote source: {SWAP_QUOTE_SOURCE}, reference date: {REFERENCE_DATE}")

## Data layer — FRED ingestion

One lightweight API call per series. The *as-of rule*: take the most recent non-missing
observation on or before the reference date (FRED reports `"."` on holidays, and publication
lags differ — e.g. `SOFR` posts next business day). Every instrument records the actual
observation date used. If the API is unreachable the notebook falls back to a hard-coded
snapshot so it still runs end-to-end offline.

In [ ]:
# =============================================================================
# SECTION 2 — FRED data layer & instrument assembly
# =============================================================================

def fred_asof(series_id: str, ref_date: str | None) -> tuple[float, str]:
    """Return (value_percent, observation_date) for the last non-missing
    observation of `series_id` on or before `ref_date` (None = latest).

    Uses sort_order=desc + a small limit so we transfer only a few rows.
    Raises on network/API errors — caller decides how to fall back."""
    params = {
        "series_id":  series_id,
        "api_key":    FRED_API_KEY,
        "file_type":  "json",
        "sort_order": "desc",     # newest first ...
        "limit":      15,         # ... a handful of rows is enough to skip holidays
    }
    if ref_date:
        params["observation_end"] = ref_date
    resp = requests.get(FRED_BASE, params=params, timeout=15)
    resp.raise_for_status()
    for obs in resp.json()["observations"]:
        if obs["value"] not in (".", ""):          # "." = holiday / not published
            return float(obs["value"]), obs["date"]
    raise ValueError(f"No usable observations for {series_id}")


def fred_history(series_id: str, start: str, end: str | None) -> pd.Series:
    """Full daily history of a series between start and end (for charts)."""
    params = {
        "series_id": series_id, "api_key": FRED_API_KEY, "file_type": "json",
        "observation_start": start,
    }
    if end:
        params["observation_end"] = end
    resp = requests.get(FRED_BASE, params=params, timeout=15)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["observations"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")   # "." -> NaN
    df["date"] = pd.to_datetime(df["date"])
    return df.dropna(subset=["value"]).set_index("date")["value"]


# --- series map --------------------------------------------------------------
SHORT_END_SERIES = {          # deposit-style nodes built from actual SOFR data
    "ON": ("SOFR",           1),    # overnight fixing; maturity ~ next business day
    "1M": ("SOFR30DAYAVG",  30),    # backward-looking compounded averages used as
    "3M": ("SOFR90DAYAVG",  90),    #   term-rate approximations (see caveats)
    "6M": ("SOFR180DAYAVG", 180),
}
TREASURY_SERIES = {y: f"DGS{y}" for y in SWAP_TENORS_Y}   # DGS1 ... DGS30

# Hard-coded offline fallback (FRED snapshot taken 2026-07-31, percent)
FALLBACK_SNAPSHOT = {
    "SOFR": 3.65, "SOFR30DAYAVG": 3.61975, "SOFR90DAYAVG": 3.62541,
    "SOFR180DAYAVG": 3.66334,
    "DGS1": 4.04, "DGS2": 4.22, "DGS3": 4.29, "DGS5": 4.37,
    "DGS7": 4.51, "DGS10": 4.67, "DGS20": 5.21, "DGS30": 5.20,
}

ref = None if REFERENCE_DATE == "latest" else REFERENCE_DATE
rows, offline = [], False

# --- short end: always from FRED SOFR series ---------------------------------
for label, (sid, days) in SHORT_END_SERIES.items():
    try:
        val, obs_date = fred_asof(sid, ref)
    except Exception as exc:                               # network/API failure
        offline, val, obs_date = True, FALLBACK_SNAPSHOT[sid], "2026-07-31"
        print(f"WARNING: FRED fetch failed for {sid} ({exc}); using snapshot.")
    rows.append({"tenor": label, "type": "deposit", "days": days,
                 "rate_pct": val, "series": sid, "obs_date": obs_date})

# --- long end: manual OIS quotes or Treasury+spread proxy --------------------
for y in SWAP_TENORS_Y:
    if SWAP_QUOTE_SOURCE == "MANUAL":
        rows.append({"tenor": f"{y}Y", "type": "swap", "years": y,
                     "rate_pct": MANUAL_SWAP_QUOTES_PCT[y],
                     "series": "manual", "obs_date": "user input"})
    elif SWAP_QUOTE_SOURCE == "TREASURY_PROXY":
        sid = TREASURY_SERIES[y]
        try:
            tsy, obs_date = fred_asof(sid, ref)
        except Exception as exc:
            offline, tsy, obs_date = True, FALLBACK_SNAPSHOT[sid], "2026-07-31"
            print(f"WARNING: FRED fetch failed for {sid} ({exc}); using snapshot.")
        quote = tsy + PROXY_SWAP_SPREAD_BP[y] / 100.0      # bp -> percent
        rows.append({"tenor": f"{y}Y", "type": "swap", "years": y,
                     "rate_pct": quote, "series": f"{sid} {PROXY_SWAP_SPREAD_BP[y]:+d}bp",
                     "obs_date": obs_date})
    else:
        raise ValueError(f"Unknown SWAP_QUOTE_SOURCE: {SWAP_QUOTE_SOURCE}")

instruments = pd.DataFrame(rows)

# Valuation date = latest observation date actually used (the curve "as of" date)
obs_dates = pd.to_datetime(instruments.loc[instruments.obs_date != "user input", "obs_date"])
VALUATION_DATE = obs_dates.max().date()
print(f"\nValuation date resolved to: {VALUATION_DATE}  "
      f"({'OFFLINE SNAPSHOT' if offline else 'live FRED data'})")
print(f"Long-end quote source: {SWAP_QUOTE_SOURCE}\n")
display(instruments[["tenor", "type", "rate_pct", "series", "obs_date"]])

# 1-year daily history of the SOFR family for the context chart (optional)
try:
    hist_start = (VALUATION_DATE - timedelta(days=365)).isoformat()
    sofr_history = pd.DataFrame({
        sid: fred_history(sid, hist_start, ref)
        for sid in ["SOFR", "SOFR30DAYAVG", "SOFR90DAYAVG", "SOFR180DAYAVG"]
    })
except Exception:
    sofr_history = None
    print("Note: history fetch failed — the history panel will be skipped.")

## Bootstrapping engine

Design (details in the methodology doc, §4):

- **State:** sorted node arrays `dates[] / dfs[]`, anchored at `(valuation date, 1.0)`.
- **Interpolation:** log-linear in $\ln P$ (piecewise-constant forwards); flat-forward
  extrapolation beyond the last node.
- **`add_swap`** treats the new node DF as the unknown `x`, prices the swap with interior
  coupon DFs interpolated *through the candidate node*, and solves the par condition with
  `brentq` (`xtol=1e-15`). With no gap this reproduces the closed-form recursion exactly, so
  the same code path handles every pillar.
- **Calendar simplifications** (documented): valuation = reference date (no T+2 spot lag),
  weekend-only roll (no US holiday calendar).

In [ ]:
# =============================================================================
# SECTION 3 — Curve engine
# =============================================================================
from calendar import monthrange

DAY_COUNT_BASIS = 360.0          # Act/360: both OIS legs
TIME_AXIS_BASIS = 365.0          # Act/365: interpolation time axis & zero rates


def add_months(d: date, months: int) -> date:
    """Calendar-month shift, clamping the day (Jan 31 + 1M -> Feb 28/29)."""
    y, m = divmod(d.year * 12 + d.month - 1 + months, 12)
    return date(y, m + 1, min(d.day, monthrange(y, m + 1)[1]))


def roll_following(d: date) -> date:
    """Roll weekend dates to the next weekday (simplified 'following' rule)."""
    while d.weekday() >= 5:                       # 5=Sat, 6=Sun
        d += timedelta(days=1)
    return d


def accrual_360(d1: date, d2: date) -> float:
    """Act/360 year fraction — the OIS day-count for both legs."""
    return (d2 - d1).days / DAY_COUNT_BASIS


class SOFRCurveBootstrapper:
    """Sequential single-curve bootstrapper for the USD SOFR OIS curve.

    Instruments MUST be added in increasing maturity order:
    deposits (ON, 1M, 3M, 6M) first, then par swaps (1Y ... 30Y).
    """

    def __init__(self, valuation_date: date):
        self.t0 = valuation_date
        self.dates = [valuation_date]      # node maturities (sorted)
        self.dfs   = [1.0]                 # discount factors, P(t0, t0) = 1

    # ---- time axis ----------------------------------------------------------
    def _t(self, d: date) -> float:
        """Act/365 year fraction from valuation date (interpolation axis)."""
        return (d - self.t0).days / TIME_AXIS_BASIS

    # ---- curve lookup -------------------------------------------------------
    def df(self, d: date, trial_node: tuple[date, float] | None = None) -> float:
        """Discount factor at date d.

        Log-linear interpolation in ln(P) between nodes; beyond the last node,
        flat-forward extrapolation (continues the last segment's slope).

        `trial_node=(maturity, x)` temporarily augments the curve with a
        candidate node — this is how the root-finder prices a swap "as if"
        the node being solved for were already on the curve."""
        dates, dfs = self.dates, self.dfs
        if trial_node is not None:
            dates, dfs = dates + [trial_node[0]], dfs + [trial_node[1]]

        t  = self._t(d)
        ts = [self._t(x) for x in dates]
        if t <= 0.0:
            return 1.0
        if t >= ts[-1]:                                   # extrapolation
            if len(ts) >= 2:
                slope = (math.log(dfs[-1]) - math.log(dfs[-2])) / (ts[-1] - ts[-2])
            else:
                slope = 0.0
            return math.exp(math.log(dfs[-1]) + slope * (t - ts[-1]))
        for i in range(1, len(ts)):                       # interpolation
            if t <= ts[i]:
                w = (t - ts[i - 1]) / (ts[i] - ts[i - 1])
                return math.exp((1.0 - w) * math.log(dfs[i - 1])
                                + w * math.log(dfs[i]))
        raise RuntimeError("unreachable")

    def _append_node(self, d: date, p: float) -> None:
        assert d > self.dates[-1], "instruments must be added in maturity order"
        assert 0.0 < p, "discount factor must be positive"
        self.dates.append(d)
        self.dfs.append(p)

    # ---- instrument bootstrap steps ----------------------------------------
    def add_deposit(self, maturity: date, rate: float) -> None:
        """Money-market node: single payoff (1 + r*tau) at maturity, Act/360.
        Used for the ON fixing and the 1M/3M/6M SOFR-average nodes.
            P(0, T) = 1 / (1 + r * tau)"""
        tau = accrual_360(self.t0, maturity)
        self._append_node(maturity, 1.0 / (1.0 + rate * tau))

    def fixed_schedule(self, years: int) -> list[date]:
        """Annual fixed-leg coupon dates: anniversaries of t0, rolled following."""
        return [roll_following(add_months(self.t0, 12 * i))
                for i in range(1, years + 1)]

    def add_swap(self, years: int, rate: float) -> None:
        """Bootstrap one par-swap pillar.

        Par condition (floating leg telescopes to 1 - P(0,Tn)):
            rate * SUM_i tau_i * P(0, T_i)  =  1 - P(0, T_n)
        Unknown: x = P(0, T_n).  Interior coupon DFs (e.g. 4Y inside the 5Y
        swap) are interpolated through the candidate node, so g(x) below is
        strictly increasing => brentq finds the unique root."""
        schedule = self.fixed_schedule(years)
        maturity = schedule[-1]

        def g(x: float) -> float:
            pv_fixed, prev = 0.0, self.t0
            for dt in schedule:
                pv_fixed += accrual_360(prev, dt) * self.df(dt, trial_node=(maturity, x))
                prev = dt
            return rate * pv_fixed - (1.0 - x)            # fixed PV - float PV

        x_star = brentq(g, 1e-6, 2.0, xtol=1e-15)
        self._append_node(maturity, x_star)

    # ---- outputs ------------------------------------------------------------
    def zero_rate(self, d: date) -> float:
        """Continuously compounded zero rate, Act/365."""
        t = self._t(d)
        return float("nan") if t <= 0 else -math.log(self.df(d)) / t

    def simple_rate(self, d: date) -> float:
        """Simple money-market zero rate, Act/360 (comparable to SOFR quotes)."""
        tau = accrual_360(self.t0, d)
        return float("nan") if tau <= 0 else (1.0 / self.df(d) - 1.0) / tau

    def forward_rate(self, d1: date, d2: date) -> float:
        """Simple forward rate between d1 and d2, Act/360."""
        tau = accrual_360(d1, d2)
        return (self.df(d1) / self.df(d2) - 1.0) / tau

    def par_rate(self, years: int) -> float:
        """Par swap rate implied by the finished curve (verification)."""
        annuity, prev = 0.0, self.t0
        schedule = self.fixed_schedule(years)
        for dt in schedule:
            annuity += accrual_360(prev, dt) * self.df(dt)
            prev = dt
        return (1.0 - self.df(schedule[-1])) / annuity


print("SOFRCurveBootstrapper defined.")

## Bootstrap execution & curve table

Deposits first (ON, 1M, 3M, 6M), then swaps in maturity order. The table reports each node's
discount factor, continuous zero rate, simple money-market zero, and the 1-year forward rate
starting at the node.

In [ ]:
# =============================================================================
# SECTION 4 — Run the bootstrap
# =============================================================================
curve = SOFRCurveBootstrapper(VALUATION_DATE)

# --- deposits (short end), in maturity order ---------------------------------
for _, row in instruments[instruments.type == "deposit"].iterrows():
    if row["tenor"] == "ON":
        maturity = roll_following(VALUATION_DATE + timedelta(days=1))
    else:
        # maturity matches the averaging window length (30/90/180 calendar days)
        maturity = roll_following(VALUATION_DATE + timedelta(days=int(row["days"])))
    curve.add_deposit(maturity, row["rate_pct"] / 100.0)   # percent -> decimal

# --- par swaps (long end), in maturity order ---------------------------------
for _, row in instruments[instruments.type == "swap"].sort_values("years").iterrows():
    curve.add_swap(int(row["years"]), row["rate_pct"] / 100.0)

# --- node table --------------------------------------------------------------
node_rows = []
tenor_labels = ["t0"] + list(instruments["tenor"])
for label, d, p in zip(tenor_labels, curve.dates, curve.dfs):
    days = (d - VALUATION_DATE).days
    node_rows.append({
        "tenor": label,
        "maturity": d.isoformat(),
        "days": days,
        "discount_factor": p,
        "zero_cont_pct": curve.zero_rate(d) * 100.0,
        # simple Act/360 zero is the money-market convention — only meaningful
        # (and only shown) out to ~1Y, where it is comparable to SOFR quotes
        "zero_mm_pct": curve.simple_rate(d) * 100.0 if 0 < days <= 370 else np.nan,
        "fwd_1y_pct": curve.forward_rate(d, add_months(d, 12)) * 100.0,
    })
curve_table = pd.DataFrame(node_rows)
print(f"Bootstrapped SOFR curve as of {VALUATION_DATE} "
      f"({len(curve.dates) - 1} nodes):\n")
display(curve_table)

## Verification — arbitrage-free repricing

The standard correctness certificate for a bootstrap: price every input instrument off the
finished curve and recover its quote. Errors should be at numerical noise level
(< 10⁻¹⁰ bp). The cell **asserts** this — it fails loudly rather than silently shipping a
mis-built curve. Two structural sanity checks are asserted as well.

In [ ]:
# =============================================================================
# SECTION 5 — Repricing verification (hard assertions)
# =============================================================================
checks = []
for _, row in instruments[instruments.type == "swap"].sort_values("years").iterrows():
    y = int(row["years"])
    implied = curve.par_rate(y) * 100.0                   # decimal -> percent
    err_bp = (implied - row["rate_pct"]) * 100.0          # percent -> bp
    checks.append({"tenor": f"{y}Y", "input_pct": row["rate_pct"],
                   "repriced_pct": implied, "error_bp": err_bp})
repricing = pd.DataFrame(checks)
shown = repricing.assign(                    # plain string formatting (no jinja2)
    input_pct=repricing.input_pct.map("{:.6f}".format),
    repriced_pct=repricing.repriced_pct.map("{:.12f}".format),
    error_bp=repricing.error_bp.map("{:.3e}".format),
)
display(shown)

worst = repricing["error_bp"].abs().max()
print(f"Worst absolute repricing error: {worst:.3e} bp")
assert worst < 1e-8, "REPRICING FAILED — curve is not arbitrage-free vs inputs"

# Structural sanity: discount factors strictly decreasing in (0, 1]
dfs = np.array(curve.dfs)
assert np.all(np.diff(dfs) < 0), "discount factors must be strictly decreasing"
assert np.all((dfs > 0) & (dfs <= 1.0)), "discount factors must lie in (0, 1]"
print("PASS — all input swaps repriced to machine precision; DF curve monotone.")

## Visualization dashboard

Four panels: (1) discount factors, (2) zero curve vs the input par quotes, (3) the 1-year
forward curve implied by log-linear DF interpolation (piecewise-constant between pillars by
construction), (4) one year of SOFR fixing/average history from FRED for context.

In [ ]:
# =============================================================================
# SECTION 6 — Charts
# =============================================================================
grid_years = np.linspace(0.02, 30.0, 601)                 # dense evaluation grid
grid_dates = [VALUATION_DATE + timedelta(days=int(round(t * 365))) for t in grid_years]
grid_df    = [curve.df(d) for d in grid_dates]
grid_zero  = [curve.zero_rate(d) * 100 for d in grid_dates]
fwd_starts = np.linspace(0.0, 29.0, 581)
grid_fwd   = [curve.forward_rate(VALUATION_DATE + timedelta(days=int(round(t * 365))),
                                 VALUATION_DATE + timedelta(days=int(round((t + 1) * 365))))
              * 100 for t in fwd_starts]

node_t = [(d - VALUATION_DATE).days / 365.0 for d in curve.dates[1:]]
node_z = [curve.zero_rate(d) * 100 for d in curve.dates[1:]]
swap_rows = instruments[instruments.type == "swap"]

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9))
fig.suptitle(f"USD SOFR Curve Bootstrap — {VALUATION_DATE}  "
             f"(quotes: {SWAP_QUOTE_SOURCE})", fontsize=13, fontweight="bold")

ax = axes[0, 0]
ax.plot(grid_years, grid_df, lw=1.8)
ax.plot(node_t, curve.dfs[1:], "o", ms=5)
ax.set_title("Discount factors  P(0, T)")
ax.set_xlabel("maturity (years)"); ax.set_ylabel("P(0, T)")

ax = axes[0, 1]
ax.plot(grid_years, grid_zero, lw=1.8, label="zero curve (cont., Act/365)")
ax.plot(node_t, node_z, "o", ms=5, label="curve nodes")
ax.plot(swap_rows["years"], swap_rows["rate_pct"], "s", ms=6, mfc="none",
        label="input par swap quotes")
ax.set_title("Zero curve vs input quotes")
ax.set_xlabel("maturity (years)"); ax.set_ylabel("rate (%)"); ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(fwd_starts, grid_fwd, lw=1.8)
ax.set_title("1Y forward rates  f(T, T+1Y)")
ax.set_xlabel("forward start (years)"); ax.set_ylabel("rate (%)")

ax = axes[1, 1]
if sofr_history is not None:
    for col in sofr_history.columns:
        ax.plot(sofr_history.index, sofr_history[col], lw=1.2, label=col)
    ax.legend(fontsize=8)
    ax.set_title("SOFR & compounded averages — trailing year (FRED)")
    ax.set_ylabel("rate (%)")
    ax.tick_params(axis="x", labelrotation=30)
else:
    ax.axis("off"); ax.text(0.5, 0.5, "history unavailable (offline)",
                            ha="center", va="center")
for a in axes.flat:
    a.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Limitations & how to harden this for production

1. **Long-end quotes.** The `TREASURY_PROXY` mode approximates SOFR OIS quotes with Treasury
   yields + a static spread table. Swap spreads move daily; for real pricing switch
   `SWAP_QUOTE_SOURCE = "MANUAL"` and paste cleared OIS quotes (CME/Bloomberg/LSEG).
2. **Short-end averages are backward-looking.** `SOFR30DAYAVG` etc. compound *realized*
   SOFR; curve nodes want *expected* SOFR. Near FOMC repricings, replace the 1M/3M nodes
   with SOFR futures-implied rates and add explicit meeting-date jump nodes.
3. **Calendar.** Weekend-only rolls and no T+2 spot lag. Production: SIFMA holiday calendar,
   modified-following, curve anchored at the spot date.
4. **Interpolation.** Log-linear DF (piecewise-flat forwards) is robust and arbitrage-safe
   but produces stepwise forwards; desks often use monotone-convex (Hagan–West) for smooth
   forwards. The engine isolates this choice in a single method (`df`) — swap it there.
5. **Single-curve is correct** for cleared SOFR swaps — but pricing legacy LIBOR-fallback
   or cross-currency positions requires additional basis curves on top of this one.

**Extending:** the finished `curve` object exposes `df()`, `zero_rate()`, `forward_rate()`
and `par_rate()` — enough to price off-market OIS positions, compute DV01 by bumping input
quotes and re-bootstrapping, or feed scenario analysis.